# Assignment RISK

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df = pd.read_excel('/Users/jlaw/projects/stern/systematic-investing/data/Assignment_RISK_data.xlsx')
df = df.drop('Time', axis=1) # Remove index which gets imported from Excel
df = df.iloc[1:] # Remove blank first row, not needed

pd.set_option('display.max_columns', None) # Show all columns when printing a dataframe

df.head()

## Correlations of Returns

#### A2-A3 have strongest absolute correlation
#### A1-A2 weakly correlated
#### A1-A3 uncorrelated

In [ ]:
df.corr()

## Cumulative Returns

#### A2-A3 correlation visible
#### A1 appears uncorrelated against both A2 and A3

In [ ]:
# Turn daily returns into cumulative returns
df[['A1_Cumulative_Returns','A2_Cumulative_Returns','A3_Cumulative_Returns']] = (1 + df[['A1','A2','A3']]).cumprod() - 1

# Plot cumulative returns
df.plot(y=['A1_Cumulative_Returns','A2_Cumulative_Returns','A3_Cumulative_Returns']);

## "Best" Performance Evaluation

### Sharpe Ratio = ( E[R] - Rf ) / Vol[R] * annualize
#### Annualized using daily returns

#### A1 has the best Sharpe Ratio, but has negative cumulative returns
#### A2 has the second best Sharpe Ratio, and has the highest cumulative returns
#### Therefore, I will recommend A2 as the "best" investment of the three

In [ ]:
# Risk free rate: annualized return of 3.7%
# 5 year T-note chosen since it is the closest to 2000 returns in the dataset
Rf_5yr_Tnote_daily = 0.037 / 252

for col in ['A1', 'A2', 'A3']:
    # Sharpe ratio = (average daily return - risk free rate) / daily volatility, annualized by multiplying by sqrt(252)
    print(f'{col}_sharpe:', ((df[col].mean() - Rf_5yr_Tnote_daily) / df[col].std()) * np.sqrt(252))

## Equal Weighted Returns

#### Equal weight yields better cumulative returns than any individual asset

In [ ]:
# Create an equal weight portfolio of the 3 assets and calculate cumulative returns
df['Equal_Weight'] = (df['A1'] + df['A2'] + df['A3']) / 3
df['Equal_Weight_Cumulative_Returns'] = (1 + df['Equal_Weight']).cumprod() - 1

# Plot cumulative returns of the 3 assets and the equal weight portfolio
df.plot(y=['A1_Cumulative_Returns','A2_Cumulative_Returns','A3_Cumulative_Returns','Equal_Weight_Cumulative_Returns']);

#### Equal weight also has a stronger Sharpe Ratio than any individual asset

In [ ]:
Rf_5yr_Tnote_daily = 0.037 / 252

# Get sharpe ratio of the equal weight portfolio
Equal_Weight_mean_daily = df['Equal_Weight'].mean()
Equal_Weight_vol_daily = df['Equal_Weight'].std()
Equal_Weight_sharpe_daily = (Equal_Weight_mean_daily - Rf_5yr_Tnote_daily) / Equal_Weight_vol_daily

Equal_Weight_sharpe = Equal_Weight_sharpe_daily * np.sqrt(252)

# Print sharpe ratio of the equal weight portfolio for comparison with the 3 individual assets
print('Equal_Weight_sharpe:', Equal_Weight_sharpe)

## Drawdowns

### Drawdown = (Value / Current Max) - 1
#### As expected, A1 has the worst drawdown (lost virtually all its value)
#### The worst drawdown of the Equal Weight portfolio was -56%


In [ ]:
# Turn cumulative returns into some real values, starting at 1 for easiest interpretation of drawdown
df[['A1_Cumulative_Value','A2_Cumulative_Value','A3_Cumulative_Value','Equal_Weight_Cumulative_Value']] = 1 + df[['A1_Cumulative_Returns','A2_Cumulative_Returns','A3_Cumulative_Returns','Equal_Weight_Cumulative_Returns']]

for col in ['A1', 'A2', 'A3', 'Equal_Weight']:
    # Current Drawdown = (Current Value / Running Maximum Value) - 1
    df[f'{col}_Drawdown'] = df[f'{col}_Cumulative_Value'] / df[f'{col}_Cumulative_Value'].cummax() - 1
    # The deepest drawdown is the minimum value of drawdowns experienced
    print(f'{col}_Drawdown:', df[f'{col}_Drawdown'].min())

## 20 Day Volatility

#### The inverse volatility weighted portfolio has less drawdown than the equal weight, but also lower cumulative returns

In [ ]:
# Calculate rolling 20-day volatility for the 3 assets
df[['A1_Vol20','A2_Vol20','A3_Vol20']] = df[['A1','A2','A3']].rolling(20).std()
# Calculate inverse of volatility for the 3 assets, which will be used as weights for the inverse volatility portfolio
df[['A1_InvVol20','A2_InvVol20','A3_InvVol20']] = 1 / df[['A1_Vol20','A2_Vol20','A3_Vol20']]

# Weight = Inverse Volatility / Sum of Inverse Volatility across the 3 assets
df[['A1_InvVol20_Weight','A2_InvVol20_Weight','A3_InvVol20_Weight']] = df[['A1_InvVol20','A2_InvVol20','A3_InvVol20']].div(df[['A1_InvVol20','A2_InvVol20','A3_InvVol20']].sum(axis=1), axis=0)
# Weights should sum to 1
df['weight_check'] = df[['A1_InvVol20_Weight','A2_InvVol20_Weight','A3_InvVol20_Weight']].sum(axis=1)

# Create inverse volatility weighted portfolio
df['InvVol_Weight'] = df['A1_InvVol20_Weight'] * df['A1'] + df['A2_InvVol20_Weight'] * df['A2'] + df['A3_InvVol20_Weight'] * df['A3']
# Calculate cumulative returns of the inverse volatility weighted portfolio
df['InvVol_Weight_Cumulative_Returns'] = (1 + df['InvVol_Weight']).cumprod() - 1

# Plot cumulative returns of the 3 assets, the equal weight portfolio and the inverse volatility weighted portfolio
df.plot(y=['A1_Cumulative_Returns','A2_Cumulative_Returns','A3_Cumulative_Returns','Equal_Weight_Cumulative_Returns','InvVol_Weight_Cumulative_Returns']);

#### The inverse volatility portfolio has significantly less drawdown than the equal weighted portfolio

In [ ]:
# Turn cumulative returns into real values, starting at 1 for easiest interpretation of drawdown
df['InvVol_Weight_Cumulative_Value'] = 1 + df['InvVol_Weight_Cumulative_Returns']
# Calculate drawdown of the inverse volatility weighted portfolio
df['InvVol_Weight_Drawdown'] = (df['InvVol_Weight_Cumulative_Value'] / df['InvVol_Weight_Cumulative_Value'].cummax()) - 1
# The deepest drawdown is the minimum value of drawdowns experienced
print('InvVol_Weight_Drawdown:', df['InvVol_Weight_Drawdown'].min())

## Predict A3 given A2

### Strategy is to have A3, but sell it when A2 volatility is higher than usual
#### There is some correlaiton between volatility of A3 and A2

In [ ]:
# Drop rows with NaN values which are created by rolling volatility calculation
df.dropna(inplace=True)
# Get correlation of the 3 assets' rolling 20-day volatility
df[['A1_Vol20','A2_Vol20','A3_Vol20']].corr()

In [ ]:
# When A2_Vol20 is greater than average, close the position in A3 (trading signal = 0), otherwise keep the position (trading signal = 1)
df['A3_Trading_Signal'] = 1
df.loc[df['A2_Vol20'] > df['A2_Vol20'].mean(), 'A3_Trading_Signal'] = 0

# Calculate strategy returns by multiplying the trading signal with the returns of A3
df['A3_Strategy_Returns'] = df['A3_Trading_Signal'] * df['A3']
# Calculate cumulative returns of the strategy
df['A3_Strategy_Cumulative_Returns'] = (1 + df['A3_Strategy_Returns']).cumprod() - 1

# Get sharpe ratio of the strategy
A3_Strategy_mean_daily = df['A3_Strategy_Returns'].mean()
A3_Strategy_vol_daily = df['A3_Strategy_Returns'].std()
A3_Strategy_sharpe_daily = (A3_Strategy_mean_daily - Rf_5yr_Tnote_daily) / A3_Strategy_vol_daily
A3_Strategy_sharpe = A3_Strategy_sharpe_daily * np.sqrt(252)
print('A3_Strategy_sharpe:', A3_Strategy_sharpe)

# Get drawdown of the strategy
df['A3_Strategy_Cumulative_Value'] = 1 + df['A3_Strategy_Cumulative_Returns']
df['A3_Strategy_Drawdown'] = (df['A3_Strategy_Cumulative_Value'] / df['A3_Strategy_Cumulative_Value'].cummax()) - 1
print('A3_Strategy_Drawdown:', df['A3_Strategy_Drawdown'].min())

# Plot cumulative returns of the strategy vs buy-and-hold A3
df.plot(y=['A3_Strategy_Cumulative_Returns', 'A3_Cumulative_Returns']);

### Clearly the strategy is ineffective
#### The cumulative returns of the strategy are lower than A3 itself
#### and the strategy has all the Drawdown of any other investment, but near-zero Sharpe Ratio